# Fragmentation Explainability Analysis

Section 18: explainability of fragmentation DAGs by chemical class.


In [ ]:
import gc
import pickle
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

DATA_ROOT = Path("../data")

## 18. Explainable Masses by Chemical Class (Fragmentation DAG Coverage)

Uses `mol_stats_df` from `data/frag/nps_nist23_d3/`, which records per-molecule fragmentation DAG statistics.

**Key metric – weighted recall (`wrecall_0.01_h1`)**: fraction of total spectral peak intensity whose *m/z* is within 0.01 Da of at least one formula mass in the fragmentation DAG (allowing up to 1 H-shift).  A low value means the DAG is missing fragments needed to explain the observed spectrum — i.e. "low explainability".

In [ ]:
# Load frag DAG stats and join with ClassyFire taxonomy
mol_stats_df = pd.read_pickle(DATA_ROOT / "frag/nps_nist23_d3/mol_stats_df.pkl")

cf_df = pd.read_pickle(DATA_ROOT / "classyfire/nist23_mol_classyfire.pkl")
cf_cols = ["mol_id", "kingdom", "superklass", "klass", "subklass"]

frag_cf = mol_stats_df.merge(cf_df[cf_cols], on="mol_id", how="left")
print(f"mol_stats_df rows: {len(mol_stats_df):,}  |  after CF join: {len(frag_cf):,}")
print(f"CF annotation coverage: {frag_cf['superklass'].notna().mean():.1%}")

WRECALL_COL = "wrecall_0.01_h1"  # primary explainability metric
print(f"\nOverall {WRECALL_COL} stats:")
print(frag_cf[WRECALL_COL].describe().round(3))

del mol_stats_df
gc.collect()


In [ ]:
# --- Weighted recall by ClassyFire Superclass ---
df_super = frag_cf.dropna(subset=["superklass"])
super_order = (
    df_super.groupby("superklass")[WRECALL_COL]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

super_stats = (
    df_super.groupby("superklass")[WRECALL_COL]
    .agg(["median", "mean", "count"])
    .reindex(super_order)
    .round(3)
)
super_stats.columns = ["median_wrecall", "mean_wrecall", "n_mols"]
print("Weighted recall by superclass:")
print(super_stats.to_string())

groups_super = [df_super[df_super["superklass"] == s][WRECALL_COL].dropna().values for s in super_order]
labels_super = [f"{s}\n(n={len(g)})" for s, g in zip(super_order, groups_super)]

fig, ax = plt.subplots(figsize=(14, 5))
bp = ax.boxplot(groups_super, labels=labels_super, patch_artist=True, medianprops=dict(color="black", linewidth=2))
for patch in bp["boxes"]:
    patch.set_facecolor("steelblue")
    patch.set_alpha(0.6)
ax.axhline(frag_cf[WRECALL_COL].median(), color="red", linestyle="--", linewidth=1.2, label="overall median")
ax.set_ylabel("Weighted Recall (wrecall_0.01_h1)")
ax.set_title("Explainable Masses by ClassyFire Superclass")
ax.set_ylim(0, 1.05)
ax.legend()
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# --- Weighted recall by ClassyFire Class (top 20 by count) ---
top20_klass = frag_cf["klass"].value_counts().head(20).index.tolist()
df_klass_fr = frag_cf[frag_cf["klass"].isin(top20_klass)].dropna(subset=["klass"])

klass_order = (
    df_klass_fr.groupby("klass")[WRECALL_COL]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

klass_stats = (
    df_klass_fr.groupby("klass")[WRECALL_COL]
    .agg(["median", "mean", "count"])
    .reindex(klass_order)
    .round(3)
)
klass_stats.columns = ["median_wrecall", "mean_wrecall", "n_mols"]
print("Weighted recall by class (top 20):")
print(klass_stats.to_string())

groups_klass = [df_klass_fr[df_klass_fr["klass"] == k][WRECALL_COL].dropna().values for k in klass_order]
labels_klass = [f"{k}\n(n={len(g)})" for k, g in zip(klass_order, groups_klass)]

fig, ax = plt.subplots(figsize=(18, 5))
bp = ax.boxplot(groups_klass, labels=labels_klass, patch_artist=True, medianprops=dict(color="black", linewidth=2))
for patch in bp["boxes"]:
    patch.set_facecolor("darkorange")
    patch.set_alpha(0.6)
ax.axhline(frag_cf[WRECALL_COL].median(), color="red", linestyle="--", linewidth=1.2, label="overall median")
ax.set_ylabel("Weighted Recall (wrecall_0.01_h1)")
ax.set_title("Explainable Masses by ClassyFire Class (top 20)")
ax.set_ylim(0, 1.05)
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [1]:
# --- Bias plot: deviation from overall median, by superclass and class ---
overall_median = frag_cf[WRECALL_COL].median()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Superclass bias
bias_super = (
    df_super.groupby("superklass")[WRECALL_COL]
    .agg(["median", "count"])
    .rename(columns={"median": "med_wrecall"})
)
bias_super["bias"] = bias_super["med_wrecall"] - overall_median
bias_super = bias_super.sort_values("bias", ascending=True)
colors_super = ["#d62728" if b < 0 else "#1f77b4" for b in bias_super["bias"]]
axes[0].barh(
    [f"{s} (n={n})" for s, n in zip(bias_super.index, bias_super["count"])],
    bias_super["bias"],
    color=colors_super,
    edgecolor="black",
    linewidth=0.5,
)
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_xlabel("Δ Weighted Recall (vs overall median)")
axes[0].set_title("Superclass Explainability Bias (wrecall_0.01_h1)")

# Class bias (top 20)
bias_klass = (
    df_klass_fr.groupby("klass")[WRECALL_COL]
    .agg(["median", "count"])
    .rename(columns={"median": "med_wrecall"})
)
bias_klass["bias"] = bias_klass["med_wrecall"] - overall_median
bias_klass = bias_klass.sort_values("bias", ascending=True)
colors_klass = ["#d62728" if b < 0 else "#1f77b4" for b in bias_klass["bias"]]
axes[1].barh(
    [f"{k} (n={n})" for k, n in zip(bias_klass.index, bias_klass["count"])],
    bias_klass["bias"],
    color=colors_klass,
    edgecolor="black",
    linewidth=0.5,
)
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("Δ Weighted Recall (vs overall median)")
axes[1].set_title("Class Explainability Bias (wrecall_0.01_h1, top 20)")

plt.tight_layout()
plt.show()

print(f"\nOverall median wrecall_0.01_h1 = {overall_median:.3f}")
print("\nLowest-explainability superclasses:")
print(bias_super.head(5)[["med_wrecall", "bias", "count"]].round(3).to_string())
print("\nLowest-explainability classes (top-20 by count):")
print(bias_klass.head(8)[["med_wrecall", "bias", "count"]].round(3).to_string())

NameError: name 'frag_cf' is not defined

In [ ]:
del frag_cf, cf_df
gc.collect()


## 18. Explainable Masses by Chemical Class (CE-Controlled)

Uses `spec_stats_df` from `data/frag/nps_nist23_d3/` (per-spectrum fragmentation DAG stats) joined
with `spec_df` for instrument/CE info and ClassyFire taxonomy.

**Metric – weighted recall (`wrecall_0.01_h1`)**: fraction of total spectral peak intensity
whose *m/z* is ≤ 0.01 Da from a formula mass in the fragmentation DAG (≤1 H-shift allowed).

**CE control**: higher collision energies produce more/different fragment ions that the DAG
may not cover, inflating 'unexplained' intensity. To avoid confounding class effects with CE
effects, we compute **CE-adjusted wrecall** = residual after subtracting the per-instrument
CE-bin median. We also show raw wrecall stratified by low/mid/high CE tertile.

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
spec_stats_df = pd.read_pickle(DATA_ROOT / "frag/nps_nist23_d3/spec_stats_df.pkl")

cf_df = pd.read_pickle(DATA_ROOT / "classyfire/nist23_mol_classyfire.pkl")

spec_meta = pd.read_pickle(DATA_ROOT / "proc/nist23/spec_df.pkl")[
    ["spec_id", "inst_type", "nce", "ace"]
]

# Merge spec-level wrecall ← spec metadata ← ClassyFire
frag_spec = (
    spec_stats_df[["spec_id", "mol_id", "wrecall_0.01_h1"]]
    .merge(spec_meta, on="spec_id", how="inner")
    .merge(cf_df[["mol_id", "superklass", "klass", "subklass"]], on="mol_id", how="left")
)

# ── Native CE per instrument (FT → NCE %, QTOF/IT → ACE eV) ─────────────────
_CE_COL = {"FT": "nce", "QTOF": "ace", "IT": "nce"}
frag_spec["native_ce"] = np.nan
for _inst, _col in _CE_COL.items():
    _m = frag_spec["inst_type"] == _inst
    frag_spec.loc[_m, "native_ce"] = frag_spec.loc[_m, _col]

# ── CE tertile bins per instrument ───────────────────────────────────────────
frag_spec["ce_bin"] = frag_spec.groupby("inst_type", observed=True)["native_ce"].transform(
    lambda x: pd.qcut(x.rank(method="first"), q=3, labels=["low", "mid", "high"])
)

# ── CE-adjusted wrecall (residual) ───────────────────────────────────────────
# Subtract per-(inst_type, ce_bin) median to isolate molecular-class effect
bin_medians = frag_spec.groupby(["inst_type", "ce_bin"], observed=True)["wrecall_0.01_h1"].transform("median")
frag_spec["wrecall_adj"] = frag_spec["wrecall_0.01_h1"] - bin_medians

print(f"Spectra: {len(frag_spec):,}  |  CF coverage: {frag_spec['superklass'].notna().mean():.1%}")
print(f"Overall median wrecall: {frag_spec['wrecall_0.01_h1'].median():.3f}")
print("\nCE bins (FT = NCE%, QTOF = ACE eV):")
print(
    frag_spec.groupby(["inst_type", "ce_bin"], observed=True)["native_ce"]
    .agg(["min", "max", "count"])
    .to_string()
)

del spec_stats_df
gc.collect()


In [ ]:
# ── CE-adjusted wrecall by Superclass ────────────────────────────────────────
df_s = frag_spec.dropna(subset=["superklass"])

super_adj = (
    df_s.groupby("superklass")[["wrecall_0.01_h1", "wrecall_adj"]]
    .agg("median")
    .rename(columns={"wrecall_0.01_h1": "med_raw", "wrecall_adj": "med_adj"})
    .join(df_s.groupby("superklass")["wrecall_0.01_h1"].count().rename("n_specs"))
    .sort_values("med_adj")
    .round(3)
)
print("Superclass — raw vs CE-adjusted median wrecall (sorted by adj, low=harder):")
print(super_adj.to_string())

In [ ]:
# ── Bar chart: raw wrecall by Superclass × CE tertile (FT only) ─────────────
df_ft = frag_spec[(frag_spec["inst_type"] == "FT") & frag_spec["superklass"].notna()].copy()

CE_COLORS = {"low": "#2196F3", "mid": "#FF9800", "high": "#F44336"}
super_order_ft = (
    df_ft.groupby("superklass")["wrecall_0.01_h1"].median().sort_values(ascending=False).index.tolist()
)

fig, ax = plt.subplots(figsize=(max(14, len(super_order_ft) * 1.3), 5))
width = 0.25
x = np.arange(len(super_order_ft))

for i, ce_lbl in enumerate(["low", "mid", "high"]):
    medians, q1s, q3s = [], [], []
    for s in super_order_ft:
        vals = df_ft[(df_ft["superklass"] == s) & (df_ft["ce_bin"] == ce_lbl)]["wrecall_0.01_h1"].dropna()
        medians.append(vals.median() if len(vals) else np.nan)
        q1s.append(vals.quantile(0.25) if len(vals) else np.nan)
        q3s.append(vals.quantile(0.75) if len(vals) else np.nan)
    medians, q1s, q3s = np.array(medians), np.array(q1s), np.array(q3s)
    ax.bar(x + i * width, medians, width=width, label=f"CE {ce_lbl}",
           color=CE_COLORS[ce_lbl], alpha=0.75, edgecolor="black", linewidth=0.4)
    ax.vlines(x + i * width, q1s, q3s, color="black", linewidth=1.0)

ax.axhline(df_ft["wrecall_0.01_h1"].median(), color="black", linestyle="--", linewidth=1, label="FT overall median")
ax.set_xticks(x + width)
ax.set_xticklabels(super_order_ft, rotation=25, ha="right")
ax.set_ylabel("Weighted Recall (wrecall_0.01_h1)")
ax.set_title("Explainable Masses by Superclass × CE Tertile (FT only, median + IQR)")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── CE-adjusted bias by Class (top 20) ──────────────────────────────────────
top20_klass = frag_spec["klass"].value_counts().head(20).index.tolist()
df_ft_klass = frag_spec[
    (frag_spec["inst_type"] == "FT") & frag_spec["klass"].isin(top20_klass)
].dropna(subset=["klass"])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (col, title, xlabel) in zip(axes, [
    ("wrecall_0.01_h1",
     "Raw Median Weighted Recall by Class",
     "Median wrecall_0.01_h1"),
    ("wrecall_adj",
     "CE-Adjusted Wrecall by Class (residual from CE-bin median)",
     "Δ Wrecall  (negative = harder than CE predicts)"),
]):
    vals = df_ft_klass.groupby("klass")[col].median().sort_values(ascending=True)
    ref = df_ft_klass["wrecall_0.01_h1"].median() if col == "wrecall_0.01_h1" else 0.0
    plot_vals = vals if col == "wrecall_adj" else vals
    colors = ["#d62728" if v < ref else "#1f77b4" for v in vals]
    bar_vals = vals.values - ref if col == "wrecall_0.01_h1" else vals.values
    ax.barh(vals.index, bar_vals, color=colors, edgecolor="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=1)
    if col == "wrecall_0.01_h1":
        ax.set_title(f"{title}\n(bars = deviation from overall median {ref:.3f})")
    else:
        ax.set_title(title)
    ax.set_xlabel(xlabel)

plt.suptitle("Explainability Bias by ClassyFire Class (top 20, FT spectra)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Print summary table
klass_summary = (
    df_ft_klass.groupby("klass")[["wrecall_0.01_h1", "wrecall_adj"]]
    .agg("median")
    .rename(columns={"wrecall_0.01_h1": "med_raw", "wrecall_adj": "med_adj"})
    .join(df_ft_klass.groupby("klass")["wrecall_0.01_h1"].count().rename("n_specs"))
    .sort_values("med_adj")
    .round(3)
)
print("Class explainability (FT, CE-adjusted, sorted by adj):")
print(klass_summary.to_string())

In [ ]:
del frag_spec
gc.collect()
